# 01 · Recolección de datos

**Objetivo.** Descargar de forma reproducible las fuentes del proyecto (precios de yfinance, hechos contables de SEC EDGAR con los reportes anuales 10-K y trimestrales 10-Q, tablas por industria de Damodaran y el `info` de yfinance), guardarlas como instantáneas con fecha y construir la serie trimestral, el EPS de los últimos 12 meses (TTM) y los 5 ratios, cada uno con su fecha de publicación.

**Punto del enunciado que cubre.** Punto 5, Recolección de datos (inventario de fuentes, modelo de entidades, enlace al diccionario y proceso de recolección).

**Preguntas que alimenta.** P1–P6 (ver la tabla del inventario, §5.1).

**Modo de ejecución.** Por defecto el notebook **no descarga**: lee las instantáneas fechadas de `data/snapshots/` y recalcula lo derivado. Con la variable de entorno `RUN_DOWNLOAD=1` vuelve a descargar; eso exige red y la variable `SEC_USER_AGENT` (identifica al solicitante ante la SEC; su valor nunca se imprime ni se guarda).

**Diccionario de datos.** Fuente única: [`docs/entrega_1/05_diccionario_datos.md`](../docs/entrega_1/05_diccionario_datos.md). Los términos financieros se explican en el [glosario](../docs/entrega_1/glosario.md).

In [1]:
import io, json, os, sys, time
from datetime import datetime, timezone
import numpy as np, pandas as pd, requests
sys.path.insert(0, "."); import comunes as cm      # nbconvert ejecuta el kernel en notebooks/, donde está comunes.py
pd.set_option("display.float_format", "{:.4f}".format)

# Por defecto NO se descarga: se usan las instantáneas fechadas de data/snapshots (reproducible y sin red).
# Para volver a descargar: RUN_DOWNLOAD=1 (requiere red y SEC_USER_AGENT en el entorno, nunca en archivos).
DESCARGAR = os.environ.get("RUN_DOWNLOAD", "0") == "1"
print("Modo:", "descarga" if DESCARGAR else "instantáneas")

# Constantes (qué significan y por qué ese valor: docs/entrega_1/04_enfoque_analitico.md §4.5)
DUR_TRIMESTRE = (80, 120)   # días de un trimestre fiscal: acepta trimestres de 12 o 16 semanas
DUR_ANUAL = (330, 400)      # días de un año fiscal: acepta años de 52 y 53 semanas
VENTANA_NORM = 12           # trimestres (3 años) del EPS promedio anual, denominador del P/E normalizado
DIAS_RECIENTE = 120         # un EDGAR cuyo último `filed` tiene más de 120 días se marca como no actualizado
PAUSA_SEC = 0.15            # segundos entre solicitudes: la SEC permite como máximo 10 por segundo
VENTANA_VECINOS = 400       # días alrededor de un cierre para comparar escala (~4 trimestres a cada lado)
MIN_VECINOS_ESCALA = 2      # vecinos publicados antes, mínimos para juzgar la escala (un error de 10³ se ve aun con 2); con menos no se corrige
TOLERANCIA_FECHA = 7        # días: dos cierres (o inicios) a menos de una semana son el mismo período reportado con otra fecha
TOL_LOG_ESCALA = 0.5        # un monto se reescala solo si es 10^(6±0,5) veces menor que sus vecinos (escrito en millones)
ACCIONES_MIN = 1e7          # acciones mínimas plausibles (la menor del universo ronda 1,2·10⁸): menos = miles o millones
ACCIONES_MAX = 1e11         # acciones máximas plausibles (la mayor ronda 8·10⁹): más = unidad equivocada

Modo: instantáneas


## 1. Instantáneas y descarga de precios (yfinance)

Cada fuente se guarda como un archivo parquet con un `.meta.json` que registra cuántas filas tiene y cuándo se descargó. Así los notebooks 02 a 04 trabajan siempre sobre la misma foto de los datos, aunque la fuente cambie después. Las funciones de abajo convierten la salida de yfinance a formato largo (una fila por fecha y símbolo).

In [2]:
COLUMNAS_PX = ["fecha", "ticker", "close", "volume"]

def guardar_snapshot(df, nombre):
    """Guarda una instantánea parquet con su fecha de descarga: el análisis es reproducible aunque la fuente cambie."""
    cm.DIR_SNAPSHOTS.mkdir(parents=True, exist_ok=True)
    df.to_parquet(cm.DIR_SNAPSHOTS / f"{nombre}.parquet", index=False)
    meta = {"nombre": nombre, "filas": len(df),
            "fecha_descarga": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    (cm.DIR_SNAPSHOTS / f"{nombre}.meta.json").write_text(json.dumps(meta, indent=2))

def a_formato_largo(raw, simbolos):
    """Salida de yf.download (columnas por símbolo) → formato largo; un símbolo sin precios se reporta como faltante."""
    partes = []
    for t in simbolos:
        if ("Close", t) not in raw.columns:
            continue
        partes.append(pd.DataFrame({"fecha": raw.index, "ticker": t, "close": raw[("Close", t)].to_numpy(),
                                    "volume": raw[("Volume", t)].to_numpy()}))
    largo = pd.concat(partes, ignore_index=True).dropna(subset=["close"]).reset_index(drop=True)
    return largo[COLUMNAS_PX], [t for t in simbolos if t not in set(largo.ticker)]

def descargar_precios(simbolos, inicio, intervalo, ajuste_dividendos=True):
    """Descarga precios; con ajuste_dividendos=False el cierre solo está ajustado por splits (precio pagado, para los múltiplos)."""
    import yfinance as yf
    raw = yf.download(simbolos, start=inicio, interval=intervalo, auto_adjust=ajuste_dividendos,
                      progress=False, group_by="column")
    return a_formato_largo(raw, simbolos)

def descargar_splits(simbolos):
    """Splits por empresa (ticker, fecha, ratio); solo aparecen las empresas que tuvieron alguno."""
    import yfinance as yf
    partes = []
    for t in simbolos:
        s = yf.Ticker(t).splits
        if len(s):
            f = pd.DatetimeIndex(s.index)
            f = (f.tz_localize(None) if f.tz is not None else f).normalize()   # fecha sin zona horaria
            partes.append(pd.DataFrame({"ticker": t, "fecha": f, "ratio": s.to_numpy(dtype=float)}))
    return pd.concat(partes, ignore_index=True)

### 1.1 Precios diarios y semanales

Se descargan las 21 empresas de la industria piloto (consumo defensivo), el SPY (mercado) y el XLP (ETF del sector): 23 símbolos. La frecuencia **diaria** alimenta la rama técnica (nivel de entrada, P4 y P6); la **semanal**, el riesgo, el sector y los múltiplos (P1, P2, P3 y P5). Los precios están ajustados por dividendos y splits (`auto_adjust=True`), que es lo que importa para el retorno total.

**Justificación.** yfinance ancla las barras semanales al día de la semana de la fecha de inicio; por eso la descarga semanal arranca el primer lunes de 2009 y todas las series comparten calendario (las uniones por fecha con el SPY y el XLP son exactas).

In [3]:
SIMBOLOS = cm.tickers() + [cm.BENCHMARK] + list(cm.ETF_SECTOR.values())     # 21 empresas + SPY + XLP
inicio_lunes = (pd.Timestamp(cm.INICIO) + pd.offsets.Week(weekday=0)).strftime("%Y-%m-%d")
if DESCARGAR:
    sem, falt_s = descargar_precios(SIMBOLOS, inicio_lunes, "1wk")
    dia, falt_d = descargar_precios(SIMBOLOS, cm.INICIO, "1d")
    print("faltantes semanal:", falt_s, "| diaria:", falt_d)
    if falt_s or falt_d:   # una descarga parcial no debe sobrescribir una instantánea completa
        raise RuntimeError(f"Descarga de precios incompleta (semanal: {falt_s}, diaria: {falt_d}); no se guarda nada")
    guardar_snapshot(sem, "precios_semanales"); guardar_snapshot(dia, "precios_diarios")
# Universo en cada carga: una instantánea vieja no contamina el análisis
precios_semanales = cm.cargar_snapshot("precios_semanales").query("ticker in @SIMBOLOS")
precios_diarios = cm.cargar_snapshot("precios_diarios").query("ticker in @SIMBOLOS")
for nombre, px in [("semanal", precios_semanales), ("diaria", precios_diarios)]:
    print(f"{nombre}: {px.ticker.nunique()} símbolos, {px.fecha.nunique()} fechas, {len(px)} filas")
print(precios_semanales.groupby("ticker").fecha.agg(["min", "max", "count"]).sort_values("min").head(5).to_string())
print("Día de la semana de las barras semanales (0 = lunes):", precios_semanales.fecha.dt.dayofweek.value_counts().to_dict())

semanal: 23 símbolos, 928 fechas, 21344 filas
diaria: 23 símbolos, 4474 fechas, 102902 filas
              min        max  count
ticker                             
ADM    2009-01-05 2026-10-12    928
TSN    2009-01-05 2026-10-12    928
SYY    2009-01-05 2026-10-12    928
STZ    2009-01-05 2026-10-12    928
SPY    2009-01-05 2026-10-12    928
Día de la semana de las barras semanales (0 = lunes): {0: 21344}


### 1.2 Splits y precio sin ajuste por dividendos (para los múltiplos)

**Justificación.** El P/E y el P/B necesitan el precio realmente pagado, no el ajustado por dividendos (que rebaja los precios históricos y sesga los múltiplos). Por eso se guarda `precios_semanales_sd` (`auto_adjust=False`), que solo está ajustado por splits, y la tabla `splits`, con la que nb04 lleva el EPS y el BVPS de EDGAR a las unidades de acciones actuales. Los retornos y el riesgo siguen usando la serie ajustada. El chequeo de abajo usa una empresa del universo con split en la muestra: alrededor del split, la serie sin ajuste por dividendos no salta a una fracción del precio.

In [4]:
if DESCARGAR:
    sem_sd, falt_sd = descargar_precios(SIMBOLOS, inicio_lunes, "1wk", ajuste_dividendos=False)
    print("faltantes sin ajuste por dividendos:", falt_sd)
    if falt_sd:            # una descarga parcial no debe sobrescribir una instantánea completa
        raise RuntimeError(f"Precios sin ajuste por dividendos incompletos: {falt_sd}; no se guarda nada")
    guardar_snapshot(descargar_splits(cm.tickers()), "splits"); guardar_snapshot(sem_sd, "precios_semanales_sd")
splits = cm.cargar_snapshot("splits").query("ticker in @cm.tickers()")
precios_semanales_sd = cm.cargar_snapshot("precios_semanales_sd").query("ticker in @SIMBOLOS")
print("splits:", len(splits), "| empresas con algún split:", sorted(splits.ticker.unique()))
# Chequeo con un split dentro de la muestra (KR, CHD o MKC, según lo que traiga `splits`)
en_muestra = splits[(splits.fecha >= cm.INICIO) & splits.ticker.isin(["KR", "CHD", "MKC"])].sort_values("fecha")
if len(en_muestra):
    t, f = en_muestra.iloc[-1][["ticker", "fecha"]]
    v = (precios_semanales_sd.query("ticker == @t")[["fecha", "close"]]
         .merge(precios_semanales.query("ticker == @t")[["fecha", "close"]], on="fecha", suffixes=("_sd", "_ajustado")))
    print(f"{t}, split {en_muestra.iloc[-1].ratio:g}:1 el {f.date()}")
    print(v[v.fecha.between(f - pd.Timedelta(weeks=3), f + pd.Timedelta(weeks=3))].to_string(index=False))

splits: 126 | empresas con algún split: ['ADM', 'CAG', 'CHD', 'CL', 'CLX', 'COST', 'GIS', 'HSY', 'KMB', 'KO', 'KR', 'MKC', 'MO', 'PEP', 'PG', 'STZ', 'SYY', 'TSN', 'WMT']
MKC, split 2:1 el 2020-12-01
     fecha  close_sd  close_ajustado
2020-11-16   91.8350         80.2020
2020-11-23   93.2400         81.4291
2020-11-30   93.9500         82.0491
2020-12-07   93.4200         81.5863
2020-12-14   93.3700         81.5426
2020-12-21   93.4700         81.6299


### 1.3 `info` de yfinance (foto del día)

El `info` trae los ratios que yfinance calcula hoy para cada empresa. No se usa para el análisis: sirve para **triangular** lo que se calcula desde EDGAR (nb02 §2b), porque los datos de un proveedor se validan antes de usarlos. Incluye `trailingEps` y `trailingPE` (EPS y P/E de los últimos 12 meses) y `sharesOutstanding` (acciones en circulación).

In [5]:
CAMPOS_INFO = ["sector", "industry", "trailingPE", "trailingEps", "priceToBook", "returnOnEquity",
               "marketCap", "sharesOutstanding"]
if DESCARGAR:
    import yfinance as yf
    filas, fallos = [], {}
    for t in cm.tickers():
        try:
            inf = yf.Ticker(t).info
            filas.append({"ticker": t, **{c: inf.get(c) for c in CAMPOS_INFO}})
        except Exception as e:                                # se registra y se sigue: no se inventa el dato
            fallos[t] = f"{type(e).__name__}: {str(e)[:80]}"
    print("fallos en info:", fallos or "ninguno")
    if fallos:             # una descarga parcial no debe sobrescribir una instantánea completa
        raise RuntimeError(f"Descarga de info incompleta: {fallos}; no se guarda nada")
    guardar_snapshot(pd.DataFrame(filas), "info_yf")
info_yf = cm.cargar_snapshot("info_yf").query("ticker in @cm.tickers()")
print("empresas:", len(info_yf), "| vacíos por campo:", info_yf.isna().sum().to_dict())

empresas: 21 | vacíos por campo: {'ticker': 0, 'sector': 0, 'industry': 0, 'trailingPE': 2, 'trailingEps': 0, 'priceToBook': 0, 'returnOnEquity': 2, 'marketCap': 0, 'sharesOutstanding': 0}


## 2. Tablas por industria (Damodaran)

Damodaran (NYU Stern) publica cada enero, en Excel y de acceso libre, promedios por industria de EE. UU. Se descargan dos tablas: **P/E** (`pedata.xls`) y **márgenes** (`margin.xls`), el referente externo con el que nb02 compara el P/E y el margen operativo calculados desde EDGAR. El enlace empresa → industria de Damodaran vive en nb02, que es el único que lo usa.

In [6]:
URLS_DAMODARAN = {"pe": "https://pages.stern.nyu.edu/~adamodar/pc/datasets/pedata.xls",
                  "margenes": "https://pages.stern.nyu.edu/~adamodar/pc/datasets/margin.xls"}

def leer_tabla_damodaran(ruta):
    """Lee la hoja de promedios por industria: busca la fila 'Industry Name' y normaliza los nombres de columna."""
    libro = pd.ExcelFile(ruta)
    hoja = "Industry Averages" if "Industry Averages" in libro.sheet_names else libro.sheet_names[0]
    crudo = pd.read_excel(libro, sheet_name=hoja, header=None)
    fila = next(i for i in range(15) if crudo.iloc[i].astype(str).str.strip().eq("Industry Name").any())
    tabla = crudo.iloc[fila + 1:].copy()
    nombres = (crudo.iloc[fila].astype(str).str.strip().str.lower()
               .str.replace(r"[^0-9a-z]+", "_", regex=True).str.strip("_"))
    # Damodaran repite nombres de columna (p. ej. para dos grupos de empresas): sufijo _2, _3...
    tabla.columns = nombres + nombres.groupby(nombres).cumcount().map(lambda n: f"_{n + 1}" if n else "")
    tabla = tabla.dropna(subset=["industry_name"]).reset_index(drop=True)
    for c in tabla.columns[1:]:
        tabla[c] = pd.to_numeric(tabla[c], errors="coerce")
    return tabla

if DESCARGAR:
    for clave, url in URLS_DAMODARAN.items():
        r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"}); r.raise_for_status()
        guardar_snapshot(leer_tabla_damodaran(io.BytesIO(r.content)), f"damodaran_{clave}")
damodaran = {c: cm.cargar_snapshot(f"damodaran_{c}") for c in URLS_DAMODARAN}
for c, t in damodaran.items():
    print(f"damodaran_{c}: {t.shape[0]} industrias, columnas {list(t.columns)[:5]}")

damodaran_pe: 96 industrias, columnas ['industry_name', 'number_of_firms', 'of_money_losing_firms_trailing', 'current_pe', 'trailing_pe']
damodaran_margenes: 96 industrias, columnas ['industry_name', 'number_of_firms', 'gross_margin', 'net_margin', 'pre_tax_pre_stock_compensation_operating_margin']


## 3. Hechos contables (SEC EDGAR)

Se usa la API `companyfacts` de la SEC (formato XBRL, taxonomía `us-gaap`): por cada empresa, todos los datos que ha reportado en sus formularios, cada uno con su período (`start`, `end`), su formulario (`form`: 10-K anual, 10-Q trimestral) y su **fecha de publicación** (`filed`). Primero se busca el número CIK de cada empresa en `company_tickers.json`. La SEC exige un `User-Agent` que identifique al solicitante (se lee de `SEC_USER_AGENT`) y como máximo 10 solicitudes por segundo.

Se guardan los conceptos de `cm.CONCEPTOS` **y además** todos los que contengan `LongTermDebt`, `OperatingIncome`, `EarningsPerShare`, `Revenue`, `SalesRevenue` o `WeightedAverageNumberOf`: así se pueden elegir nombres alternativos con los datos, sin una segunda descarga. El filtro es por concepto, no por formulario, así que los 10-Q quedan incluidos.

In [7]:
URL_TICKERS = "https://www.sec.gov/files/company_tickers.json"
URL_FACTS = "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"

def descargar_json(url):
    """GET a la SEC con el User-Agent del entorno (requisito de la SEC) y una pausa para no pasar de 10 solicitudes/s."""
    agente = os.environ.get("SEC_USER_AGENT")
    if not agente:
        raise RuntimeError("Defina SEC_USER_AGENT en el entorno (requisito de la SEC)")
    time.sleep(PAUSA_SEC)
    r = requests.get(url, headers={"User-Agent": agente}, timeout=30); r.raise_for_status()
    return r.json()

def hechos_a_df(companyfacts):
    """JSON de companyfacts → tabla larga: una fila por hecho (concepto, unidad, período, valor, formulario, publicación)."""
    filas = [{"concepto": concepto, "unidad": unidad, "start": h.get("start"), "end": h.get("end"),
              "val": h.get("val"), "form": h.get("form"), "fp": h.get("fp"), "filed": h.get("filed")}
             for concepto, cont in companyfacts.get("facts", {}).get("us-gaap", {}).items()
             for unidad, hechos in cont.get("units", {}).items() for h in hechos]
    df = pd.DataFrame(filas, columns=["concepto", "unidad", "start", "end", "val", "form", "fp", "filed"])
    for c in ("start", "end", "filed"):
        df[c] = pd.to_datetime(df[c])
    return df

if DESCARGAR:
    ciks = {v["ticker"]: str(v["cik_str"]).zfill(10) for v in descargar_json(URL_TICKERS).values()}
    nombres = [n for alt, _ in cm.CONCEPTOS.values() for n in alt]
    partes, errores = [], {}
    for t in cm.tickers():                                        # los ETF no tienen companyfacts
        try:
            h = hechos_a_df(descargar_json(URL_FACTS.format(cik=ciks[t])))
            h = h[h.concepto.isin(nombres) | h.concepto.str.contains("LongTermDebt|OperatingIncome|EarningsPerShare|Revenue|SalesRevenue|WeightedAverageNumberOf")]
            partes.append(h.assign(ticker=t))
        except Exception as e:
            errores[t] = f"{type(e).__name__}: {str(e)[:80]}"
    print("errores:", errores or "ninguno")
    if errores:            # una descarga parcial no debe sobrescribir una instantánea completa
        raise RuntimeError(f"Descarga de EDGAR incompleta para {sorted(errores)}; no se guarda nada")
    guardar_snapshot(pd.concat(partes, ignore_index=True), "edgar_hechos")
edgar_hechos = cm.cargar_snapshot("edgar_hechos")
hechos = edgar_hechos[edgar_hechos.ticker.isin(cm.tickers())]       # nunca tickers de una instantánea vieja
print("empresas:", hechos.ticker.nunique(), "de", len(cm.tickers()), "| hechos:", len(hechos))
print("Formularios:", hechos.form.value_counts().to_dict())
frescura = hechos.groupby("ticker").filed.max().rename("ultimo_filed").to_frame()
frescura["dias"] = (pd.Timestamp.now().normalize() - frescura.ultimo_filed).dt.days
frescura["no_actualizado"] = frescura.dias > DIAS_RECIENTE
print(frescura.sort_values("dias", ascending=False).to_string())

empresas: 21 de 21 | hechos: 72682
Formularios: {'10-Q': 44922, '10-K': 25223, '8-K': 1191, '10-Q/A': 584, '10-K/A': 583, '10-KT': 129, 'DEF 14A': 40, 'PRE 14A': 10}
       ultimo_filed  dias  no_actualizado
ticker                                   
MDLZ     2026-04-28   170            True
KO       2026-04-30   168            True
PM       2026-07-24    83           False
HSY      2026-07-30    77           False
MO       2026-07-30    77           False
CHD      2026-07-31    76           False
CL       2026-07-31    76           False
TSN      2026-08-03    73           False
ADM      2026-08-04    72           False
KMB      2026-08-04    72           False
PG       2026-08-28    48           False
WMT      2026-08-28    48           False
KR       2026-09-18    27           False
GIS      2026-09-23    22           False
CAG      2026-09-30    15           False
MKC      2026-10-01    14           False
SYY      2026-10-01    14           False
COST     2026-10-07     8           

**Frescura de EDGAR.** La tabla de arriba marca las empresas cuyo último dato publicado tiene más de 120 días. Para KO y MDLZ, los datos de la SEC (`companyfacts`) terminan en los reportes publicados a fines de abril de 2026 (el 2026-04-30 y el 2026-04-28), aunque esas empresas ya publicaron reportes posteriores. Es una limitación de la fuente (temporalidad: la SEC aún no incorporó esos reportes a `companyfacts`), no un error del código. Su EPS TTM más reciente es el del primer trimestre de 2026, y nb02 lo tiene en cuenta al compararlo con yfinance.

### 3.1 Nombres alternativos de ingresos, deuda y acciones

Una empresa puede reportar el mismo concepto con otro nombre XBRL, o cambiarlo con los años. Revisando los datos de cada empresa se agregaron a `cm.CONCEPTOS`, después de los nombres principales (que tienen prioridad), nombres que miden lo mismo o, en patrimonio y utilidad neta, una aproximación que incluye a los socios minoritarios:

| Concepto | Nombre agregado | Empresas que lo usan | Qué mide |
|---|---|---|---|
| Ingresos | `SalesRevenueGoodsNet` | CAG, CHD, GIS, KO, KR, STZ y SYY (hasta 2018) | Ventas netas de productos, el nombre que usaban antes de la norma contable de ingresos de 2018 |
| Deuda de largo plazo | `LongTermDebtAndCapitalLeaseObligations` | TSN, SYY, PM, KMB, MDLZ y PEP | Deuda de largo plazo no corriente, incluidos los arrendamientos financieros |
| Acciones | `WeightedAverageNumberOfShareOutstandingBasicAndDiluted` | MO (2011–2020) y PM (2012–2016) | Acciones promedio cuando las básicas y las diluidas son iguales |
| EPS reportado | `EarningsPerShareBasicAndDiluted` | MO (2011–2020) | EPS cuando el básico y el diluido son iguales |
| Patrimonio | `StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest` | ver el conteo de §3.2 | **Aproximación:** patrimonio total, incluye la parte de los socios minoritarios, así que sobrestima un poco el del accionista común (afecta P/B y ROE) |
| Utilidad neta | `ProfitLoss` | ver el conteo de §3.2 | **Aproximación:** utilidad consolidada, incluye la de los minoritarios |

**Lo que no se puede cubrir (completitud, se cuenta en nb02):**
- ADM y CLX no reportan utilidad operativa con ningún nombre. Solo reportan la utilidad antes de impuestos, que no es lo mismo: no tienen margen operativo y no se inventa un sustituto.
- STZ y HSY reportan las acciones y el EPS por clase de acción. Ese desglose no viene en los datos agregados de la SEC (`companyfacts`), así que STZ no tiene EPS y HSY solo lo tiene hasta 2015.

La celda lista, por empresa, los nombres de ingresos y de deuda que existen en los datos y los años que cubren.

**Justificación.** Un concepto ausente se deja vacío (nunca 0); un nombre alternativo solo entra si mide el mismo concepto, salvo las dos aproximaciones con minoritarios, que se aceptan como respaldo (solo cuando falta el nombre principal) y se declaran con su conteo.

In [8]:
def cobertura_nombres(h, patron, empresas=None):
    """Por empresa y nombre XBRL que contiene `patron`: primer y último año fiscal con dato anual en un 10-K."""
    d = h[h.concepto.str.contains(patron) & (h.form == "10-K") & (h.fp == "FY")]
    if empresas is not None:
        d = d[d.ticker.isin(empresas)]
    return (d.assign(anio=d["end"].dt.year).groupby(["ticker", "concepto"]).anio
            .agg(["min", "max", "nunique"]).rename(columns={"nunique": "anios"}))

pd.set_option("display.max_rows", 200)
print("Ingresos: nombres con 'Revenue' por empresa")
print(cobertura_nombres(hechos, "Revenue").to_string())
print("\nDeuda de largo plazo: nombres con 'LongTermDebt' (PM, SYY y TSN)")
print(cobertura_nombres(hechos, "LongTermDebt", ["PM", "SYY", "TSN"]).to_string())
print("\nAlternativas en uso:", {k: v[0] for k, v in cm.CONCEPTOS.items() if k in ("ingresos", "deuda_lp")})

Ingresos: nombres con 'Revenue' por empresa
                                                                                                                       min   max  anios
ticker concepto                                                                                                                        
ADM    ContractWithCustomerLiabilityRevenueRecognized                                                                 2018  2025      8
       CostOfRevenue                                                                                                  2008  2025     18
       EquityMethodInvestmentSummarizedFinancialInformationNetSalesOrGrossRevenue                                     2008  2010      3
       EquityMethodInvestmentSummarizedFinancialInformationRevenue                                                    2009  2019     11
       RelatedPartyTransactionRevenuesFromTransactionsWithRelatedParty                                                2008  2011      4
    

### 3.2 Tabla anual y los 5 ratios, cada uno con su fecha de publicación

De cada 10-K se toma el valor anual de los conceptos (ingresos, utilidad operativa, utilidad neta, patrimonio, deuda de largo plazo, acciones diluidas promedio y EPS reportado), con la **primera publicación** de cada año (lo que el inversor vio en su momento). Con ellos se calculan los 5 ratios del [glosario](../docs/entrega_1/glosario.md): EPS (utilidad por acción), BVPS (valor en libros por acción), ROE, margen operativo y deuda/patrimonio.

**Fecha por ratio:** cada ratio se usa desde que se publicaron **sus** dos conceptos, no desde el último de la fila ([notebook 02 §3](02_calidad.ipynb)). **Acciones implícitas:** si faltan las acciones diluidas (pasa con empresas que reportan por clase de acción, como STZ y HSY), se usan utilidad neta ÷ EPS reportado, con la fecha del más tardío de los dos; la columna `acciones_implicitas` las marca.

In [9]:
def escala_acciones(v):
    """#11 para ACCIONES (regla absoluta, sin vecinos ni futuro): ninguna empresa del universo tiene menos de
    ACCIONES_MIN ni más de ACCIONES_MAX acciones. Un valor menor está en miles (×10³) o en millones (×10⁶);
    uno mayor, en una unidad 10³ veces más chica (÷10³). Devuelve (valor corregido, ¿se corrigió?)."""
    v = np.asarray(v, dtype=float).copy(); c = np.zeros(len(v), bool)
    for f in (1e6, 1e3):                                         # millones (KO, MDLZ) y miles (CLX, HSY)
        m = (v > 0) & (v < ACCIONES_MIN) & (v * f >= ACCIONES_MIN) & (v * f <= ACCIONES_MAX)
        v[m] *= f; c |= m
    m = v > ACCIONES_MAX; v[m] /= 1e3; c |= m
    return v, c

def serie_anual(h, alternativas, unidad):
    """Un valor anual (10-K, año fiscal) por cierre, con su PRIMERA publicación; un concepto ausente da tabla vacía (nunca ceros)."""
    d = h[h.concepto.isin(alternativas) & (h.unidad == unidad) & (h.form == "10-K") & (h.fp == "FY")].copy()
    if d.empty:
        return pd.DataFrame({"end": pd.Series(dtype="datetime64[us]"), "filed": pd.Series(dtype="datetime64[us]"),
                             "val": pd.Series(dtype="float64"), "concepto": pd.Series(dtype=object)})
    dur = (d["end"] - d["start"]).dt.days
    d = d[d["start"].isna() | dur.between(*DUR_ANUAL)]           # saldos (sin start) o flujos de un año completo
    d = d[d["filed"] >= d["end"]]                                  # publicado antes del cierre: imposible
    d["prioridad"] = d["concepto"].map({c: i for i, c in enumerate(alternativas)})
    d = d.sort_values(["end", "filed", "prioridad"]).drop_duplicates("end", keep="first")
    if unidad == "shares":                                         # #11: acciones en miles o millones (HSY 2011)
        d["val"] = escala_acciones(d["val"])[0]
    return d[["end", "filed", "val", "concepto"]].reset_index(drop=True)

In [10]:
# Cada ratio se usa desde que se publicaron SUS conceptos (no desde el último de la fila).
# Los conceptos y los ratios viven en comunes.py porque nb02 también los usa.

def tabla_anual(h):
    """Una fila por cierre fiscal: cada concepto con SU fecha de publicación (filed_<concepto>)."""
    partes = []
    for nombre, (alternativas, unidad) in cm.CONCEPTOS.items():
        s = serie_anual(h, alternativas, unidad).rename(columns={"val": nombre, "filed": f"filed_{nombre}"})
        partes.append(s.drop(columns="concepto").set_index("end"))
    t = pd.concat(partes, axis=1, sort=True).sort_index()
    for nombre in cm.CONCEPTOS:                    # concepto ausente → NaN/NaT, nunca 0
        t[nombre] = t.get(nombre, np.nan); t[f"filed_{nombre}"] = t.get(f"filed_{nombre}", pd.NaT)
    # Si faltan las acciones, acciones implícitas = utilidad neta ÷ EPS reportado, fechadas con el más tardío
    imp = t["acciones"].isna() & t["utilidad_neta"].notna() & t["eps_reportado"].notna() & (t["eps_reportado"] != 0)
    t.loc[imp, "acciones"] = t["utilidad_neta"] / t["eps_reportado"]
    t.loc[imp, "filed_acciones"] = t[["filed_utilidad_neta", "filed_eps_reportado"]].max(axis=1)
    t["acciones_implicitas"] = imp                 # nb02 las cuenta
    return t.reset_index().rename(columns={"index": "end"})

def ratios_anuales(t):
    """Los 5 ratios; cada uno con la publicación más tardía de sus DOS conceptos (NaT si falta uno)."""
    r = pd.DataFrame({"end": t["end"]})
    for ratio, (num, den) in cm.RATIOS.items():
        r[ratio] = t[num] / t[den].where(t[den] > 0 if den == "patrimonio" else t[den] != 0)   # dominio: patrimonio > 0
        f = t[[f"filed_{num}", f"filed_{den}"]]
        r[f"filed_{ratio}"] = f.max(axis=1).where(f.notna().all(axis=1))
    return r

tabla = pd.concat([tabla_anual(h).assign(ticker=t) for t, h in hechos.groupby("ticker")], ignore_index=True)
ratios = pd.concat([ratios_anuales(tabla_anual(h)).assign(ticker=t) for t, h in hechos.groupby("ticker")], ignore_index=True)

Las tablas derivadas se guardan **solo si cambiaron**: así, al ejecutar el notebook sin descarga, ni los archivos ni su fecha se reescriben, y el historial del repositorio muestra solo cambios reales.

In [11]:
def normalizar_tipos(df):
    """Fechas en microsegundos (la resolución con la que pandas 3 lee el parquet): evita que equals falle solo por ns frente a us."""
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].astype("datetime64[us]")
    return df

def guardar_si_cambio(df, nombre, orden=("ticker", "end")):
    """Guarda una instantánea DERIVADA solo si su contenido cambió; así, sin descarga, ni el archivo ni su fecha se reescriben."""
    orden = list(orden)
    nuevo = normalizar_tipos(df.sort_values(orden).reset_index(drop=True))
    ruta = cm.DIR_SNAPSHOTS / f"{nombre}.parquet"
    if ruta.exists():
        actual = normalizar_tipos(cm.cargar_snapshot(nombre).sort_values(orden).reset_index(drop=True))
        if list(actual.columns) == list(nuevo.columns) and actual.equals(nuevo):
            print(f"{nombre}: sin cambios; se conserva la instantánea existente")
            return
    guardar_snapshot(nuevo, nombre)
    print(f"{nombre}: guardada ({len(nuevo)} filas)")

guardar_si_cambio(tabla, "tabla_anual"); guardar_si_cambio(ratios, "ratios_anuales")
print(ratios.groupby("ticker").size().describe().round(1).to_dict())
print("Rezago mediano de publicación del EPS (días):", (ratios.filed_eps - ratios.end).dt.days.median())
cob = tabla.groupby("ticker").agg(anios=("end", "size"), con_eps=("acciones", lambda s: s.notna().sum()),
                                  implicitas=("acciones_implicitas", "sum"), con_deuda=("deuda_lp", lambda s: s.notna().sum()))
print("Cobertura anual por empresa (años con acciones para el EPS, cuántos con acciones implícitas, años con deuda):")
print(cob.to_string())
# Cuántos años-empresa usan un nombre alternativo (no el principal) en la tabla anual, por concepto y nombre
uso = pd.concat([serie_anual(h, alt, u).assign(ticker=t, nombre=n) for n, (alt, u) in cm.CONCEPTOS.items()
                 for t, h in hechos.groupby("ticker")], ignore_index=True)
uso = uso[uso.concepto != uso.nombre.map(lambda n: cm.CONCEPTOS[n][0][0])]
print("Años-empresa con un nombre alternativo:")
print(uso.groupby(["nombre", "concepto"]).agg(anios=("end", "size"), empresas=("ticker", lambda x: ", ".join(sorted(set(x))))).to_string())

tabla_anual: sin cambios; se conserva la instantánea existente
ratios_anuales: sin cambios; se conserva la instantánea existente
{'count': 21.0, 'mean': 19.7, 'std': 0.6, 'min': 19.0, '25%': 19.0, '50%': 20.0, '75%': 20.0, 'max': 21.0}
Rezago mediano de publicación del EPS (días): 52.0
Cobertura anual por empresa (años con acciones para el EPS, cuántos con acciones implícitas, años con deuda):
        anios  con_eps  implicitas  con_deuda
ticker                                       
ADM        21       19           0         18
CAG        20       19           1         17
CHD        19       18           0         17
CL         19       19           1         18
CLX        19       18           0         17
COST       20       19           0         18
GIS        20       19           1         18
HSY        20        7           0         18
KMB        19       19           1         17
KO         20       19           0         18
KR         20       19           0         18
MDLZ 

Años-empresa con un nombre alternativo:
                                                                                      anios                                                                          empresas
nombre        concepto                                                                                                                                                       
acciones      WeightedAverageNumberOfShareOutstandingBasicAndDiluted                      8                                                                            MO, PM
              WeightedAverageNumberOfSharesOutstandingBasic                               3                                                                               HSY
deuda_lp      LongTermDebtAndCapitalLeaseObligations                                     94                                     CL, GIS, HSY, KMB, KO, KR, MDLZ, PM, SYY, TSN
              LongTermDebtNoncurrent                                                     4

## 4. Reportes trimestrales (10-Q) y EPS de los últimos 12 meses (TTM)

Una empresa publica 4 reportes al año: 3 trimestrales (10-Q) y 1 anual (10-K). Si solo se usara el 10-K, el P/E usaría utilidades de hasta un año atrás. Con el **EPS de los últimos 12 meses (TTM)**, la suma de los 4 trimestres más recientes, el P/E se actualiza en cada reporte, como lo mira un inversor real. La historia empieza en 2009 porque antes no hay datos estructurados gratuitos (el XBRL se volvió obligatorio entre 2009 y 2011): es una limitación.

Los datos trimestrales traen 11 problemas. Así se procesan:

| # | Problema | Ejemplo | Qué se hace | Dónde se revisa |
|---|---|---|---|---|
| 1 | Acumulados del año | El 10-Q de junio trae la utilidad de 6 meses | Se usa el dato de ~90 días; si solo hay acumulado, trimestre = acumulado − acumulado anterior (`origen = resta_ytd`). Si el 10-Q trae los dos, gana el de ~90 días; la tabla de orígenes de §4.3 muestra cuántas restas hicieron falta (pueden ser 0) | [diccionario](../docs/entrega_1/05_diccionario_datos.md) (`tabla_trimestral.origen`), [nb02 §2b](02_calidad.ipynb) |
| 2 | No hay 10-Q del cuarto trimestre | El Q4 lo cubre el 10-K | Q4 = año − (Q1 + Q2 + Q3), publicado con el 10-K (`q4_derivado`) | [nb02 §2b](02_calidad.ipynb) |
| 3 | Años fiscales distintos del calendario | WMT cierra en enero | Todo se alinea por fecha de cierre (`end`) y de publicación (`filed`), nunca por "Q1–Q4" | [diccionario](../docs/entrega_1/05_diccionario_datos.md) |
| 4 | Un trimestre reaparece corregido en reportes posteriores | Los 10-Q repiten el año anterior para comparar | Se usa la primera publicación (lo que se sabía entonces), sin formularios corregidos (/A); entre un dato reportado y uno derivado gana el que se conoció primero | [nb02 §2b](02_calidad.ipynb) |
| 5 | Flujos frente a saldos | La utilidad se acumula; el patrimonio es una foto a una fecha | Suma de 4 trimestres solo para utilidad neta, ingresos y utilidad operativa; último valor para patrimonio y deuda | [diccionario](../docs/entrega_1/05_diccionario_datos.md) (`ratios_ttm`) |
| 6 | El número de acciones cambia | Recompras | Acciones del último trimestre (en el cierre del año, el promedio anual del 10-K); si faltan, acciones implícitas (utilidad neta ÷ EPS reportado) | [nb02 §2b](02_calidad.ipynb) |
| 7 | Trimestres faltantes | Un concepto cambia de nombre un año | El TTM queda vacío, nunca 0 | [nb02 §2b](02_calidad.ipynb) |
| 8 | Trimestres de 12 o 16 semanas | Comercio minorista, años de 53 semanas | Se aceptan de 80 a 120 días | [diccionario](../docs/entrega_1/05_diccionario_datos.md) |
| 9 | El TTM mezcla 4 reportes | Q1 a Q4 se publican en fechas distintas | Existe desde que se publicó el último de los 4; vence a los 548 días (nb04) | [diccionario](../docs/entrega_1/05_diccionario_datos.md) (`filed_eps_ttm`) |
| 10 | TTM negativo o cero | Un año con pérdidas | Se conserva, pero el P/E queda vacío | [nb02 §2b](02_calidad.ipynb) |
| 11 | Escala inconsistente | KO y MDLZ reportaron acciones en millones en algunos 10-Q; CLX y HSY, en miles; KMB, la utilidad del Q1 2009–10 en unidades | **Acciones:** ninguna empresa del universo tiene menos de 10 millones de acciones, así que un valor menor se lleva a unidades (×10³ o ×10⁶), también en el dato anual. **Montos:** se comparan con datos del mismo concepto publicados antes (ya corregidos) y solo se reescalan si son ~10⁶ veces menores (escritos en millones); una utilidad real cercana a cero (100 a 1000 veces menor) no se toca. Sin al menos 2 vecinos previos no se corrige y se marca (`escala_sin_vecinos`). Lo corregido se marca (`escala_corregida`) | [nb02 §2b](02_calidad.ipynb) |

**P/E normalizado.** Además se calcula el **EPS promedio de 3 años** (`eps_prom_3a`): el promedio anual de las utilidades por acción de los últimos 3 años: la utilidad neta de los 12 trimestres más recientes ya publicados, dividida entre 3 y entre las acciones del último trimestre (se divide entre las acciones de hoy para que un split no distorsione el promedio). Es el denominador del P/E normalizado de P2 y P3. Un año de utilidades excepcionales puede hacer que una acción parezca barata sin serlo; promediar varios años de utilidades evita esa ilusión (Graham y Dodd, 1934; Campbell y Shiller, 1998, DOI [10.3905/jpm.24.2.11](https://doi.org/10.3905/jpm.24.2.11)). Se usan 3 años, y no 5–10, porque los datos estructurados empiezan en 2009 (limitación).

### 4.1 Serie trimestral de los flujos

Se toman los hechos de 10-Q y 10-K de cada concepto, se corrige la escala (#11) y se conserva la primera publicación de cada período (#4). El EPS reportado se procesa como un flujo (resta de acumulados y Q4 derivado); es una aproximación, porque el EPS de cada trimestre usa su propio promedio de acciones, y solo se usa para las acciones implícitas. Las acciones son un promedio y no se pueden restar: en el cierre del año fiscal se usa el promedio anual del 10-K.

In [12]:
# Parte 1: serie trimestral de los FLUJOS (utilidad neta, ingresos, utilidad operativa, acciones y EPS reportado).
FLUJOS = ["utilidad_neta", "ingresos", "utilidad_operativa", "acciones", "eps_reportado"]
SALDOS = ["patrimonio", "deuda_lp"]

def primera_publicacion(d, claves):
    """#4: un período aparece de nuevo en reportes posteriores (comparativos o reexpresados).
    Se conserva la PRIMERA publicación (filed mínimo): lo que el inversor vio en su momento (sin look-ahead).
    Entre nombres XBRL publicados el mismo día gana el de mayor prioridad (orden de cm.CONCEPTOS)."""
    d = d[d["filed"] >= d["end"]]                                  # publicado antes del cierre: imposible
    return d.sort_values(["filed", "prioridad"]).drop_duplicates(claves, keep="first")

def hechos_de(h, nombre):
    """Hechos de un concepto (con sus alternativas XBRL) en 10-Q y 10-K, nunca 8-K ni formularios corregidos (/A)."""
    alternativas, unidad = cm.CONCEPTOS[nombre]
    d = h[h.concepto.isin(alternativas) & (h.unidad == unidad) & h.form.isin(["10-Q", "10-K"])].copy()
    d["prioridad"] = d["concepto"].map({c: i for i, c in enumerate(alternativas)})
    return corregir_escala(unificar_fechas(d))

def unificar_fechas(d):
    """#3: un mismo período a veces aparece con el cierre corrido 1 a 3 días (CHD 2010-04-02 en su 10-Q y 2010-03-31
    en los comparativos de un 10-K posterior). Las fechas a ±TOLERANCIA_FECHA días se tratan como el mismo
    cierre (o inicio) y toman la del dato publicado primero; así no se crean trimestres duplicados."""
    d = d.copy()
    for c in ("end", "start"):
        x = d[d[c].notna()].sort_values([c, "filed"])
        grupo = (x[c].diff().dt.days > TOLERANCIA_FECHA).cumsum()          # cierres a ≤ 7 días = mismo grupo
        canon = x.sort_values("filed").groupby(grupo)[c].transform("first")   # la fecha del primero publicado
        d.loc[x.index, c] = canon
    return d

def corregir_escala(d):
    """#11: errores de escala XBRL (KO y MDLZ reportaron acciones en millones; CLX y HSY, en miles; KMB, la
    utilidad en unidades). Se reescalan y se marcan con `escala_corregida`."""
    d = d.copy(); d["escala_corregida"] = False; d["escala_sin_vecinos"] = False
    val = d["val"].to_numpy(dtype=float).copy(); corr = np.zeros(len(d), bool); sinv = np.zeros(len(d), bool)
    if (d["unidad"] == "shares").all():                          # acciones: regla absoluta (ver escala_acciones)
        val, corr = escala_acciones(val)
        d["val"] = val; d["escala_corregida"] = corr
        return d
    # Montos en USD: se comparan con la mediana de sus VECINOS = mismo nombre XBRL, cierre a ±VENTANA_VECINOS días
    # y publicados ANTES que el valor (filed < su filed), ya corregidos y en orden de publicación: solo se usa lo
    # que se sabía entonces (sin look-ahead) y un error repetido no arrastra la mediana.
    # Solo se reescala por 10⁶ (monto escrito en millones sin su escala, KMB 2009–10) y si el cociente
    # mediana ÷ |valor| está cerca de 10⁶ (|log10(cociente) − 6| < TOL_LOG_ESCALA). No se reescala por 10³: un
    # trimestre real cercano al punto de equilibrio puede ser 100 a 1000 veces menor que lo típico (ADM 2009-03:
    # 3 M, cociente 10^2,4; SYY 2020-03: −3 M, 10^2,2; STZ 2021-08: 1,5 M, 10^2,8) y no se distingue de un error
    # de miles. Las acciones en miles se corrigen con la regla absoluta. Con menos de MIN_VECINOS_ESCALA vecinos no se corrige y se marca.
    fin = d["end"].to_numpy(); fil = d["filed"].to_numpy(); nom = d["concepto"].to_numpy()
    for i in np.argsort(fil, kind="stable"):
        v = (nom == nom[i]) & (np.abs((fin - fin[i]).astype("timedelta64[D]").astype(int)) <= VENTANA_VECINOS) & (fil < fil[i])
        if v.sum() < MIN_VECINOS_ESCALA:
            sinv[i] = True; continue
        med = np.median(np.abs(val[v]))
        if val[i] == 0 or med == 0:
            continue
        if abs(np.log10(med / abs(val[i])) - 6) < TOL_LOG_ESCALA:
            val[i] *= 1e6; corr[i] = True
    d["val"] = val; d["escala_corregida"] = corr; d["escala_sin_vecinos"] = sinv
    return d

def unir(partes, cols):
    """pandas 3: concatenar solo piezas no vacías y fijar los tipos (fechas y float)."""
    partes = [x for x in partes if len(x)]
    t = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame(columns=cols)
    for c in ("start", "end", "filed"):
        if c in t: t[c] = pd.to_datetime(t[c])
    if "val" in t: t["val"] = t["val"].astype(float)
    for c in ("escala_corregida", "escala_sin_vecinos"):
        if c in t: t[c] = t[c].fillna(False).astype(bool)
    return t

In [13]:
PRIORIDAD_ORIGEN = {"reportado": 0, "resta_ytd": 1, "q4_derivado": 2}   # desempate con el mismo `filed`

def primero_conocido(q):
    """#4: por cada `end` gana el dato publicado primero; si dos tienen el mismo `filed`, gana el reportado
    (luego la resta de acumulados y al final el Q4 derivado). Orden estable: el resultado no depende del azar."""
    q = q.assign(_p=q["origen"].map(PRIORIDAD_ORIGEN))
    q = q.sort_values(["filed", "_p"], kind="stable").drop_duplicates("end", keep="first")
    return q.drop(columns="_p")

def trimestres_flujo(h, nombre):
    """Un valor por trimestre fiscal (alineado por `end`, nunca por año calendario ni por "Q1–Q4", #3)."""
    d = hechos_de(h, nombre).dropna(subset=["start"])
    d["dur"] = (d["end"] - d["start"]).dt.days
    cols = ["start", "end", "val", "filed", "form", "escala_corregida", "escala_sin_vecinos"]
    # #1 y #8: dato de duración trimestral (80–120 días: acepta trimestres de 12 o 16 semanas y años de 53)
    q = primera_publicacion(d[d.dur.between(*DUR_TRIMESTRE)], ["end"])[cols].assign(origen="reportado")
    if nombre == "acciones":
        # Las acciones son un PROMEDIO, no se pueden restar. Para el cierre del año fiscal (sin 10-Q) se usa
        # el promedio anual del 10-K, el dato publicado más cercano (#6; se declara como precisión en nb02).
        a = primera_publicacion(d[d.dur.between(*DUR_ANUAL) & (d.form == "10-K")], ["end"])
        a = a[~a["end"].isin(q["end"])][cols].assign(origen="promedio_anual")
        return unir([q, a], cols + ["origen"]).sort_values("end")
    # #1: si solo hay acumulado del año (YTD de ~180 o ~270 días), trimestre = acumulado − acumulado anterior
    #     del MISMO año fiscal (misma `start`); Q1 es el acumulado de 3 meses de ese año.
    acum = primera_publicacion(d[d.dur.between(DUR_TRIMESTRE[0], DUR_ANUAL[0] - 1)], ["start", "end"])
    nuevos = []
    for s, g in acum.sort_values("end").groupby("start"):
        g = g.reset_index(drop=True)
        for i in range(1, len(g)):
            act, ant = g.iloc[i], g.iloc[i - 1]
            salto = (act["end"] - ant["end"]).days
            if not DUR_TRIMESTRE[0] <= salto <= DUR_TRIMESTRE[1]:
                continue                                   # los acumulados no son consecutivos
            nuevos.append({"start": ant["end"] + pd.Timedelta(days=1), "end": act["end"],
                           "val": act["val"] - ant["val"], "filed": max(act["filed"], ant["filed"]),  # el más tardío
                           "form": act["form"], "origen": "resta_ytd"})
    # #4: entre reportado y derivado del mismo `end`, gana el que se conoció primero (menor `filed`)
    q = unir([q, pd.DataFrame(nuevos, columns=cols + ["origen"])], cols + ["origen"])
    q = primero_conocido(q)
    # #2: no hay 10-Q del cuarto trimestre. Q4 = anual (10-K, 330–400 días) − suma de los 3 trimestres
    #     de ESE año fiscal (los que terminan dentro del año, por `end`); su `filed` es el del 10-K.
    anual = primera_publicacion(d[d.dur.between(*DUR_ANUAL) & (d.form == "10-K")], ["end"])
    q4 = []
    for _, a in anual.iterrows():
        dentro = q[(q["end"] > a["start"]) & (q["end"] < a["end"] - pd.Timedelta(days=DUR_TRIMESTRE[0] - 1))]
        if len(dentro) != 3:
            continue                                       # sin los 3 trimestres no se deriva Q4: queda vacío (#7)
        q4.append({"start": dentro["end"].max() + pd.Timedelta(days=1), "end": a["end"],
                   "val": a["val"] - dentro["val"].sum(), "filed": max(a["filed"], dentro["filed"].max()),
                   "form": "10-K", "origen": "q4_derivado"})
    q = unir([q, pd.DataFrame(q4, columns=cols + ["origen"])], cols + ["origen"])
    return primero_conocido(q).sort_values("end")   # #4: gana el que se conoció primero

### 4.2 Saldos y tabla trimestral

El patrimonio y la deuda son saldos: una foto en la fecha de cierre, sin `start`. No se suman; se toma el valor de cada cierre con su primera publicación. La tabla trimestral queda en formato largo (una fila por empresa, concepto y cierre) con el origen de cada valor, para auditarla en nb02.

In [14]:
# Parte 2: saldos y tabla trimestral (formato largo).
def trimestres_saldo(h, nombre):
    """#5: el patrimonio y la deuda son saldos (una foto a una fecha, sin `start`): no se suman.
    Se toma el valor al cierre de cada período, con su primera publicación (#4)."""
    d = hechos_de(h, nombre)
    d = primera_publicacion(d[d["start"].isna()], ["end"])
    return d[["start", "end", "val", "filed", "form", "escala_corregida", "escala_sin_vecinos"]].assign(origen="saldo")

def tabla_trimestral(h):
    """Los 7 conceptos por trimestre, cada valor con SU `filed` y su origen (para auditar en nb02)."""
    partes = [trimestres_flujo(h, c).assign(concepto=c) for c in FLUJOS]
    partes += [trimestres_saldo(h, c).assign(concepto=c) for c in SALDOS]
    return unir(partes, ["start", "end", "val", "filed", "form", "escala_corregida", "escala_sin_vecinos", "origen", "concepto"])   # tipos fijos

### 4.3 TTM, P/E normalizado y los 5 ratios trimestrales

El TTM de cada flujo es la suma de los 4 últimos trimestres **consecutivos**; si falta uno, queda vacío. El EPS TTM usa las acciones del último trimestre. El EPS promedio de 3 años es la utilidad neta de los 12 trimestres más recientes, dividida entre 3 y entre las acciones del último trimestre: así queda en la misma escala que el EPS TTM, y un split dentro de esos 3 años no lo distorsiona. Cada ratio lleva la fecha en que se publicó el último de sus propios datos: solo se usa desde entonces. Si el patrimonio es cero o negativo (pasa con empresas que recompraron más acciones que su patrimonio, como PM, MO o CL), el ROE y la deuda/patrimonio quedan vacíos: no se pueden leer como ratio, igual que un P/E con utilidad negativa. La tabla `ratios_ttm` es la fuente del P/E y del P/B de nb04.

In [15]:
# Parte 3: TTM y los 5 ratios, cada uno con su fecha de disponibilidad.
def ttm_flujo(tq, concepto):
    """#5 y #7: TTM = suma de los 4 ÚLTIMOS trimestres CONSECUTIVOS hasta cada `end`.
    Si falta uno (o no son consecutivos) el TTM queda NaN: nunca se rellena con ceros.
    #9: su `filed` es el más tardío de los 4 (existe desde que se publicó el último)."""
    s = tq[tq.concepto == concepto].sort_values("end").reset_index(drop=True)
    filas = []
    for i in range(3, len(s)):
        v = s.iloc[i - 3:i + 1]
        dias = (v["end"].iloc[-1] - v["end"].iloc[0]).days        # 4 trimestres seguidos ≈ 273 días entre cierres
        ok = 3 * DUR_TRIMESTRE[0] <= dias <= 3 * DUR_TRIMESTRE[1] and v["val"].notna().all()
        filas.append({"end": v["end"].iloc[-1], f"{concepto}_ttm": v["val"].sum() if ok else np.nan,
                      f"filed_{concepto}_ttm": v["filed"].max() if ok else pd.NaT})
    t = pd.DataFrame(filas, columns=["end", f"{concepto}_ttm", f"filed_{concepto}_ttm"])
    t[f"filed_{concepto}_ttm"] = pd.to_datetime(t[f"filed_{concepto}_ttm"])   # pandas 3: fecha aunque todo sea vacío
    t[f"{concepto}_ttm"] = t[f"{concepto}_ttm"].astype(float)                   # y número aunque la empresa no reporte el concepto (ADM, CLX)
    return t.set_index("end")

def ultimo_valor(tq, concepto):
    """#5 y #6: saldos y acciones en el cierre de cada trimestre (el último valor publicado para ese `end`)."""
    s = tq[tq.concepto == concepto].set_index("end")
    return s[["val", "filed"]].rename(columns={"val": concepto, "filed": f"filed_{concepto}"})

def eps_promedio(tq, n=VENTANA_NORM):
    """Promedio ANUAL de las utilidades por acción de los últimos 3 años (denominador del P/E normalizado):
    suma de la utilidad neta de los n trimestres consecutivos hasta cada `end` ÷ (n/4) años ÷ acciones del ÚLTIMO
    trimestre de la ventana. Se divide entre las acciones de hoy (no entre las de cada trimestre) para que un split
    dentro de la ventana no distorsione el promedio; es la misma convención del EPS TTM (#6).
    Vacío si falta un trimestre de utilidad o las acciones del último (nunca 0). Su `filed` es el más tardío
    de los n trimestres y de esas acciones: existe desde que se publicó el último."""
    u = tq[tq.concepto == "utilidad_neta"].set_index("end")[["val", "filed"]].sort_index()
    a = tq[tq.concepto == "acciones"].set_index("end")[["val", "filed"]]
    span = pd.Series(u.index, index=u.index).diff(n - 1).dt.days      # n trimestres seguidos ≈ (n−1)·91 días
    ok = span.between((n - 1) * DUR_TRIMESTRE[0], (n - 1) * DUR_TRIMESTRE[1])
    suma = u["val"].rolling(n, min_periods=n).sum()
    f_u = pd.to_datetime(u["filed"].astype("datetime64[ns]").astype("int64").rolling(n, min_periods=n).max())
    acc = a["val"].reindex(u.index); f_a = a["filed"].reindex(u.index).astype("datetime64[ns]")
    out = pd.DataFrame({"eps_prom_3a": (suma / (n / 4) / acc.where(acc > 0)).where(ok),
                        "filed_eps_prom_3a": pd.concat([f_u, f_a], axis=1).max(axis=1, skipna=False).where(ok)})
    out.loc[out.eps_prom_3a.isna(), "filed_eps_prom_3a"] = pd.NaT
    out.index = pd.to_datetime(out.index); return out

In [16]:
def ratios_ttm(tq):
    """Una fila por cierre de trimestre: flujos TTM, saldos y acciones, y los 5 ratios con filed_<ratio>."""
    r = pd.concat([ttm_flujo(tq, c) for c in ["utilidad_neta", "ingresos", "utilidad_operativa", "eps_reportado"]],
                  axis=1).sort_index()
    r.index = pd.to_datetime(r.index); r.index.name = "end"         # pandas 3: índice con tipo fecha
    for c in SALDOS + ["acciones"]:
        # saldo o acciones: el último valor con `end` ≤ cierre del trimestre (merge_asof hacia atrás, sin futuro).
        # Tolerancia de un trimestre (120 días): si el saldo de ese trimestre falta, el ratio queda vacío (#7)
        # en lugar de arrastrar un valor de años atrás que haría parecer disponible un dato que no existe.
        u = ultimo_valor(tq, c).sort_index().reset_index()
        r = pd.merge_asof(r.reset_index().sort_values("end"), u, on="end", direction="backward",
                          tolerance=pd.Timedelta(days=DUR_TRIMESTRE[1])).set_index("end")
    # #6: si faltan las acciones del trimestre, acciones implícitas = utilidad neta TTM ÷ EPS reportado TTM
    imp = r["acciones"].isna() & r["utilidad_neta_ttm"].notna() & r["eps_reportado_ttm"].notna() & (r["eps_reportado_ttm"] != 0)
    r.loc[imp, "acciones"] = r["utilidad_neta_ttm"] / r["eps_reportado_ttm"]
    r.loc[imp, "filed_acciones"] = r[["filed_utilidad_neta_ttm", "filed_eps_reportado_ttm"]].max(axis=1)
    r["acciones_implicitas"] = imp
    def ratio(num, den, nombre):
        """Cada ratio existe desde la publicación más tardía de SUS dos insumos (NaT si falta uno)."""
        # Dominio: con patrimonio ≤ 0 (recompras mayores que el patrimonio) el ROE y la deuda/patrimonio no se
        # pueden leer como ratio (cambian de signo o explotan): quedan vacíos, igual que el P/E con EPS ≤ 0
        r[nombre] = r[num] / r[den].where(r[den] > 0 if den == "patrimonio" else r[den] != 0)
        f = r[[f"filed_{num}", f"filed_{den}"]].where(r[nombre].notna())
        r[f"filed_{nombre}"] = f.max(axis=1).where(f.notna().all(axis=1))
    ratio("utilidad_neta_ttm", "acciones", "eps_ttm")             # #6: EPS TTM con las acciones del último trimestre
    ratio("patrimonio", "acciones", "bvps")
    ratio("utilidad_neta_ttm", "patrimonio", "roe_ttm")
    ratio("utilidad_operativa_ttm", "ingresos_ttm", "margen_operativo_ttm")
    ratio("deuda_lp", "patrimonio", "deuda_patrimonio")
    r = r.join(eps_promedio(tq), how="left")                       # P/E normalizado (P2 y P3)
    r.index = pd.to_datetime(r.index)                               # pandas 3: tipos fijos
    for c in [c for c in r.columns if c.startswith("filed_")]:
        r[c] = pd.to_datetime(r[c])
    return r.drop(columns=["eps_reportado_ttm", "filed_eps_reportado_ttm"]).reset_index()

In [17]:
tq = pd.concat([tabla_trimestral(h).assign(ticker=t) for t, h in hechos.groupby("ticker")], ignore_index=True)
rt = pd.concat([ratios_ttm(tq[tq.ticker == t]).assign(ticker=t) for t in tq.ticker.unique()], ignore_index=True)
guardar_si_cambio(tq, "tabla_trimestral", orden=["ticker", "concepto", "end"])   # solo si cambió
guardar_si_cambio(rt, "ratios_ttm")
print(tq.groupby(["concepto", "origen"]).size().unstack(fill_value=0))   # cuántos trimestres por origen
print("Restas de acumulados por empresa:", tq[tq.origen == "resta_ytd"].groupby("ticker").size().to_dict())
print("Trimestres con EPS TTM por empresa:", rt.dropna(subset=["eps_ttm"]).groupby("ticker").size().describe().round(1).to_dict())
print("TTM ≤ 0 (P/E vacío, #10):", int((rt.eps_ttm <= 0).sum()))
print("Patrimonio ≤ 0 (ROE y deuda/patrimonio vacíos por dominio):", rt[rt.patrimonio <= 0].groupby("ticker").size().to_dict(), "| total:", int((rt.patrimonio <= 0).sum()))
print("Hechos con escala corregida (#11):", tq[tq.escala_corregida].groupby(["ticker", "concepto"]).size().to_dict())
print("Trimestres sin vecinos previos para revisar la escala (no se corrigen):", tq[tq.escala_sin_vecinos].groupby("ticker").size().to_dict())
print("Trimestres con EPS promedio de 3 años por empresa:", rt.dropna(subset=["eps_prom_3a"]).groupby("ticker").size().describe().round(1).to_dict())

tabla_trimestral: sin cambios; se conserva la instantánea existente
ratios_ttm: sin cambios; se conserva la instantánea existente
origen              promedio_anual  q4_derivado  reportado  resta_ytd  saldo
concepto                                                                    
acciones                       319            0       1061          0      0
deuda_lp                         0            0          0          0   1419
eps_reportado                    0          137       1221          2      0
ingresos                         0          147       1349          0      0
patrimonio                       0            0          0          0   1499
utilidad_neta                    0          140       1357          0      0
utilidad_operativa               0          230       1087          0      0
Restas de acumulados por empresa: {'KMB': 2}
Trimestres con EPS TTM por empresa: {'count': 20.0, 'mean': 65.4, 'std': 9.6, 'min': 25.0, '25%': 67.0, '50%': 67.0, '75%': 68.2, 'm

**Ejemplo para el lector.** KO cierra su año fiscal en diciembre y WMT en enero. La tabla muestra los últimos 8 trimestres de la utilidad neta de cada una, con su origen: un `q4_derivado` es el cuarto trimestre que se obtiene restando los tres primeros al dato anual, y un `resta_ytd` (si aparece), un trimestre obtenido restando dos acumulados del año.

In [18]:
for t in ["KO", "WMT"]:
    v = tq[(tq.ticker == t) & (tq.concepto == "utilidad_neta")].sort_values("end").tail(8)
    print(t); print(v[["start", "end", "val", "filed", "origen"]].to_string(index=False))

KO
     start        end             val      filed      origen
2024-03-30 2024-06-28 2411000000.0000 2024-07-29   reportado
2024-06-29 2024-09-27 2848000000.0000 2024-10-24   reportado
2024-09-28 2024-12-31 2195000000.0000 2025-02-20 q4_derivado
2025-01-01 2025-03-28 3330000000.0000 2025-05-01   reportado
2025-03-29 2025-06-27 3810000000.0000 2025-07-24   reportado
2025-06-28 2025-09-26 3696000000.0000 2025-10-23   reportado
2025-09-27 2025-12-31 2271000000.0000 2026-02-20 q4_derivado
2026-01-01 2026-04-03 3924000000.0000 2026-04-30   reportado
WMT
     start        end             val      filed      origen
2024-08-01 2024-10-31 4577000000.0000 2024-12-06   reportado
2024-11-01 2025-01-31 5254000000.0000 2025-03-14 q4_derivado
2025-02-01 2025-04-30 4487000000.0000 2025-06-06   reportado
2025-05-01 2025-07-31 7026000000.0000 2025-08-29   reportado
2025-08-01 2025-10-31 6143000000.0000 2025-12-03   reportado
2025-11-01 2026-01-31 4237000000.0000 2026-03-13 q4_derivado
2026-02-01 2026-0

## 5. Fuentes

### 5.1 Inventario

| Fuente | URL | Estructura | Granularidad | Cobertura | Acceso y términos | Importancia | Preguntas |
|---|---|---|---|---|---|---|---|
| **yfinance (precios)** | https://finance.yahoo.com (paquete `yfinance`) | Serie de tiempo en panel (símbolo × fecha) | Diaria y semanal; `close` ajustado y sin ajuste por dividendos, y `volume` | 23 símbolos (21 empresas de consumo defensivo, SPY y XLP) desde 2009 | Gratuito, sin llave; uso personal o educativo, no es una API oficial | **Crítica** | P1, P4 y P6 (diaria/semanal), P2 y P3 (precio del múltiplo), P5 (SPY y XLP) |
| **SEC EDGAR `companyfacts`** | https://data.sec.gov/api/xbrl/companyfacts/ | Hechos contables en formato largo (concepto × período × `filed`) | Trimestral (10-Q) y anual (10-K), con fecha de publicación | 21 empresas desde 2009 | Dominio público; `User-Agent` identificado y máximo 10 solicitudes por segundo | **Crítica** | P2 y P3 (P/E normalizado y P/E TTM), lectura fundamental del producto |
| **Damodaran Online** | https://pages.stern.nyu.edu/~adamodar/ | Tabla de referencia (una fila por industria) | Anual (enero) | Industrias de EE. UU.; se usan P/E y márgenes | Gratuito para uso académico, con cita | Complementaria | Validación del P/E y del margen (nb02); referente de la lectura fundamental |
| **yfinance `info`** | https://finance.yahoo.com (`Ticker.info`) | Tabla de referencia (una fila por empresa) | Foto del día de descarga | 21 empresas | Igual que los precios; los campos cambian sin aviso | Complementaria | Triangulación del EPS TTM y del P/E (nb02 §2b) |

Una fuente es **crítica** si sin ella no se puede responder ninguna pregunta principal, y **complementaria** si solo sirve para validar o dar contexto. Las columnas de cada instantánea están en el [diccionario de datos](../docs/entrega_1/05_diccionario_datos.md).

### 5.2 Modelo de entidades

```mermaid
erDiagram
    EMPRESA ||--o{ PRECIO_DIARIO : "ticker"
    EMPRESA ||--o{ PRECIO_SEMANAL : "ticker"
    EMPRESA ||--o{ HECHO_CONTABLE : "ticker"
    EMPRESA ||--o{ TRIMESTRE : "ticker"
    EMPRESA ||--o{ RATIO_TTM : "ticker"
    EMPRESA ||--o{ RATIO_ANUAL : "ticker"
    EMPRESA }o--|| ETF : "sector"
    ETF ||--o{ PRECIO_DIARIO : "ticker"
    ETF ||--o{ PRECIO_SEMANAL : "ticker"
    HECHO_CONTABLE ||--o{ TRIMESTRE : "se procesa en"
    TRIMESTRE ||--o{ RATIO_TTM : "4 trimestres"
    HECHO_CONTABLE ||--o{ RATIO_ANUAL : "10-K"
    EMPRESA {
        string ticker PK
        string industria
    }
    PRECIO_DIARIO {
        string ticker FK
        date fecha
        float close
        float volume
    }
    PRECIO_SEMANAL {
        string ticker FK
        date fecha
        float close
        float volume
    }
    HECHO_CONTABLE {
        string ticker FK
        string concepto
        date end
        date filed
        string form
        float val
    }
    TRIMESTRE {
        string ticker FK
        string concepto
        date end
        date filed
        string origen
        float val
    }
    RATIO_TTM {
        string ticker FK
        date end
        float eps_ttm
        float eps_prom_3a
        date filed_eps_ttm
    }
    RATIO_ANUAL {
        string ticker FK
        date end
        float eps
        date filed_eps
    }
    ETF {
        string ticker PK
        string industria
    }
```

Los precios y los datos contables se unen por empresa, y en el tiempo por la fecha de publicación (`filed_<ratio>`), no por el cierre fiscal: así nada en una fecha usa datos que todavía no se conocían (sin *look-ahead*). La industria de cada empresa sale de `cm.UNIVERSO` y su ETF de `cm.ETF_SECTOR`.

### 5.3 Volumen e integridad de las instantáneas

La tabla sale de los metadatos de cada instantánea y del tamaño de cada archivo; después se comprueba que las llaves de cada tabla no estén duplicadas.

In [19]:
filas = []
for meta in sorted(cm.DIR_SNAPSHOTS.glob("*.meta.json")):
    m = json.loads(meta.read_text()); pq = cm.DIR_SNAPSHOTS / f"{m['nombre']}.parquet"
    filas.append({"instantanea": m["nombre"], "filas": m["filas"], "columnas": pd.read_parquet(pq).shape[1],
                  "kb": pq.stat().st_size / 1024, "fecha_descarga_utc": m["fecha_descarga"]})
volumen = pd.DataFrame(filas)
print(volumen.to_string(index=False))
print("Total de filas:", volumen.filas.sum(), "| tamaño total (MB):", round(volumen.kb.sum() / 1024, 2))
# ¿Las llaves esperadas son únicas? (edgar_hechos puede repetir un hecho; la primera publicación se queda con uno)
for nombre, llaves in [("precios_diarios", ["fecha", "ticker"]), ("precios_semanales", ["fecha", "ticker"]),
                       ("tabla_trimestral", ["ticker", "concepto", "end"]), ("ratios_ttm", ["ticker", "end"]),
                       ("ratios_anuales", ["ticker", "end"]),
                       ("edgar_hechos", ["ticker", "concepto", "unidad", "start", "end", "filed", "form", "fp"])]:
    d = cm.cargar_snapshot(nombre)
    print(f"{nombre:18s} filas={len(d):>7d} llaves duplicadas={d.duplicated(llaves).sum()}")

         instantanea  filas  columnas        kb        fecha_descarga_utc
  damodaran_margenes     96        19   29.1855 2026-10-15T20:39:20+00:00
        damodaran_pe     96        10   15.5654 2026-10-15T20:39:20+00:00
        edgar_hechos  72682         9  445.1436 2026-10-15T20:39:54+00:00
             info_yf     21         9    6.9736 2026-10-15T20:39:19+00:00
     precios_diarios 102902         4 1517.2900 2026-10-15T20:38:58+00:00
   precios_semanales  21344         4  326.4834 2026-10-15T20:38:58+00:00
precios_semanales_sd  21344         4  279.2031 2026-10-15T20:39:09+00:00
      ratios_anuales    414        12   38.7031 2026-10-16T01:14:57+00:00
          ratios_ttm   1434        27  258.7158 2026-10-16T01:15:01+00:00
              splits    126         3    3.4189 2026-10-15T20:39:09+00:00
         tabla_anual    414        17   49.6084 2026-10-16T00:55:01+00:00
    tabla_trimestral   9968        10  107.5596 2026-10-16T01:15:01+00:00
Total de filas: 230841 | tamaño total 

**Síntesis**
- Las instantáneas cubren 23 símbolos (21 empresas de consumo defensivo, SPY y XLP) en frecuencia diaria y semanal, y los hechos de EDGAR de las 21 empresas en 10-Q y 10-K; las cifras están en las salidas de las celdas.
- De los 10-Q y 10-K salen la tabla trimestral, el EPS TTM y el EPS promedio de 3 años (P/E normalizado), con los 11 problemas de los datos trimestrales resueltos sin rellenar vacíos con ceros.
- Cada ratio tiene su fecha de publicación, así que los notebooks siguientes solo usan lo que ya se conocía en cada fecha.
- La calidad de estos datos (completitud, triangulación con yfinance y Damodaran, casos de escala) se mide en el [notebook 02](02_calidad.ipynb).